# Лабораторная работа №3 — Морфологические операции и детекция автомобилей
**Тема:** выделение движущихся автомобилей на видео со статической камеры методом вычитания фона.

**Выполнение:** все пять частей методички: усреднённый фон за 30 секунд, абсолютная разность и порог, dilate → erode, 8-связные компоненты и bounding boxes, сохранение видео, CPU-бенчмарк, сравнение с MOG2.

**Важно:** требуется собственный ролик дорожного движения с **неподвижной камерой**, длительностью **более 30 секунд**. Видео не входит в репозиторий. Запустите блоки по порядку в Google Colab (CPU достаточно). Результаты и выводы рассчитываются по вашему видео, а не выдумываются заранее.

In [ ]:
import os, time, math
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Video
print("OpenCV:", cv2.__version__)
cv2.setNumThreads(1)  # воспроизводимый CPU-бенчмарк на одном потоке

## 1. Загрузка видео и построение опорного кадра
Загрузите **свой** MP4/AVI с фиксированной дорожной камеры (не образец из методички). Если файл уже лежит в Colab, укажите его путь в `VIDEO_PATH`. При необходимости подключите Google Drive и укажите полный путь. Начальные 30 секунд используются **только** для построения фона; оценка идёт по последующим кадрам.

In [ ]:
VIDEO_PATH = ""  # например "/content/drive/MyDrive/traffic.mp4"; пусто = загрузить вручную
if not VIDEO_PATH:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Загрузите видео или задайте VIDEO_PATH")
    VIDEO_PATH = next(iter(uploaded))
VIDEO_PATH = str(Path(VIDEO_PATH))
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Не удалось открыть видео: {VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if not np.isfinite(fps) or fps <= 0: raise ValueError("Неверный FPS видео")
BG_SECONDS = 30
bg_frames = int(round(BG_SECONDS * fps))
if total > 0 and total <= bg_frames:
    raise ValueError(f"Видео слишком короткое: {total/fps:.1f} с. Нужно более 30 с для анализа.")
print(f"Видео: {width}×{height}, {fps:.2f} FPS, кадров={total}, длительность≈{total/fps:.1f} с")
assert width > 0 and height > 0

In [ ]:
def rgb_show(img, title="", ax=None):
    if ax is None: _, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img,
              cmap=None if img.ndim == 3 else "gray", vmin=None if img.ndim == 3 else 0,
              vmax=None if img.ndim == 3 else 255)
    ax.set_title(title); ax.axis("off")
    return ax

# Стриминговое попиксельное среднее: память O(ширина × высота), а не O(30 × FPS × кадр).
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
acc = None; actual_bg_frames = 0; first_frame = None
for i in range(bg_frames):
    ok, frame = cap.read()
    if not ok: break
    if first_frame is None: first_frame = frame.copy()
    if acc is None: acc = np.zeros_like(frame, dtype=np.float64)
    acc += frame.astype(np.float64)
    actual_bg_frames += 1
if actual_bg_frames < bg_frames:
    cap.release()
    raise ValueError(f"Прочитано лишь {actual_bg_frames}/{bg_frames} фоновых кадров. Нужно 30 с.")
background = np.rint(acc / actual_bg_frames).clip(0, 255).astype(np.uint8)
cv2.imwrite("background.png", background)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rgb_show(first_frame, "Первый кадр", axes[0])
rgb_show(background, f"Среднее за {BG_SECONDS} секунд ({actual_bg_frames} кадров)", axes[1])
plt.tight_layout(); plt.show()
print("Фон сохранён: background.png")

Формула усреднения: \(B(x,y)=\frac1N\sum_{i=1}^{N}I_i(x,y)\). Кратковременные автомобили размываются, неподвижные элементы сохраняются. Машина, стоящая на месте большую часть интервала, может остаться в модели фона.

## 2–3. Порог, морфология, связные компоненты
\(D_t(x,y)=|I_t(x,y)-B(x,y)|\), маска \(M=255\) при \(\mathrm{gray}(D)>T\), иначе 0.

Морфологическое **закрытие**: сначала расширение (`dilate`), затем сужение (`erode`) тем же ядром. Связные компоненты ищем по 8-связности, отбрасываем площади менее 100 пикселей. Рамки обозначают **движение**, а не гарантированно автомобили (могут попасть пешеходы и тени).

In [ ]:
THRESHOLD = 50
KERNEL_SIZE = 5  # нечётный положительный размер
MIN_AREA = 100

def detect_static(frame, background, threshold=50, kernel_size=5, min_area=100, stages=False):
    if kernel_size < 1 or kernel_size % 2 == 0: raise ValueError("kernel_size должен быть нечётным >=1")
    timings = {}
    t = time.perf_counter()
    diff = cv2.absdiff(frame, background)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    timings["difference_gray"] = time.perf_counter() - t

    t = time.perf_counter()
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    timings["threshold"] = time.perf_counter() - t

    t = time.perf_counter()
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    dilated = cv2.dilate(mask, kernel, iterations=1)
    closed = cv2.erode(dilated, kernel, iterations=1)
    timings["morphology"] = time.perf_counter() - t

    t = time.perf_counter()
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(closed, connectivity=8)
    keep = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]
    clean = (np.isin(labels, keep).astype(np.uint8) * 255)
    boxes = [tuple(int(v) for v in stats[i, :5]) for i in keep]  # x,y,w,h,area
    timings["components_filter"] = time.perf_counter() - t

    t = time.perf_counter()
    annotated = frame.copy()
    for x,y,w,h,area in boxes:
        cv2.rectangle(annotated, (x,y), (x+w-1,y+h-1), (0,255,0), 2)
    timings["drawing"] = time.perf_counter() - t
    result = dict(annotated=annotated, mask=clean, boxes=boxes, timings=timings)
    if stages: result.update(diff_gray=gray, binary=mask, dilated=dilated, closed=closed)
    return result

In [ ]:
def read_at(frame_index):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()
    if not ok: raise RuntimeError(f"Не прочитан кадр {frame_index}")
    return frame

# Берём кадры спустя 1, 3 и 6 секунд после фонового отрезка (если доступны).
sample_indices = [bg_frames + int(round(s * fps)) for s in (1, 3, 6)]
sample_indices = [i for i in sample_indices if total <= 0 or i < total]
if not sample_indices: sample_indices = [bg_frames]
sample_frames = [read_at(i) for i in sample_indices]
sample_results = [detect_static(f, background, THRESHOLD, KERNEL_SIZE, MIN_AREA, True) for f in sample_frames]
for idx, frame, r in zip(sample_indices, sample_frames, sample_results):
    fig, axs = plt.subplots(2, 4, figsize=(18, 8))
    for ax, img, title in zip(axs.flat,
         [frame, background, r["diff_gray"], r["binary"], r["dilated"], r["closed"], r["mask"], r["annotated"]],
         ["Кадр", "Фон", "|Кадр − фон|, gray", "Бинаризация", "Dilation", "Erosion (closing)", "Компоненты ≥100", f"Рамки: {len(r['boxes'])}"]):
        rgb_show(img, title, ax)
    fig.suptitle(f"Кадр {idx}, время {idx/fps:.1f} с", fontsize=14)
    plt.tight_layout(); plt.show()
    print(f"Кадр {idx}: площади компонент {[b[4] for b in r['boxes'][:20]]}")

### Влияние параметров
Сравним пороги, размеры ядра и варианты морфологии на **одном и том же** кадре. Большее ядро закрывает дырки, но может соединить соседние машины; высокий порог сокращает ложные срабатывания, но теряет слабо отличающиеся объекты.

In [ ]:
frame = sample_frames[0]
diff = cv2.cvtColor(cv2.absdiff(frame, background), cv2.COLOR_BGR2GRAY)
fig, axs = plt.subplots(3, 4, figsize=(15, 10))
for row, t in enumerate([25, 50, 75]):
    _, raw = cv2.threshold(diff, t, 255, cv2.THRESH_BINARY)
    for col, k in enumerate([1, 3, 5, 9]):
        kernel = np.ones((k,k), np.uint8)
        cleaned = cv2.erode(cv2.dilate(raw, kernel), kernel)
        rgb_show(cleaned, f"Порог={t}, ядро={k}", axs[row,col])
plt.tight_layout(); plt.show()

_, raw = cv2.threshold(diff, THRESHOLD, 255, cv2.THRESH_BINARY)
kernel = np.ones((KERNEL_SIZE, KERNEL_SIZE), np.uint8)
variants = {
    "Без морфологии": raw,
    "Только dilate": cv2.dilate(raw, kernel),
    "Только erode": cv2.erode(raw, kernel),
    "Closing (dilate→erode)": cv2.erode(cv2.dilate(raw,kernel),kernel),
    "Opening (erode→dilate)": cv2.dilate(cv2.erode(raw,kernel),kernel),
}
fig, axs = plt.subplots(1,5, figsize=(19,4))
for ax, (name, mask) in zip(axs, variants.items()): rgb_show(mask, name, ax)
plt.tight_layout(); plt.show()

## 4. Обработка видео и анализ CPU-производительности
Измеряем только сам алгоритм (без чтения кадра с диска, записи MP4 и построения графиков). Для MOG2 ниже применяется тот же принцип. При необходимости измените `FRAME_STEP` на 10 для ускоренного **анализа каждого десятого кадра**, однако выходное видео тогда будет содержать только обработанные кадры; его FPS скорректирован для сохранения исходного темпа.

In [ ]:
FRAME_STEP = 1  # 1 = каждый кадр, 10 = каждый десятый
MAX_EVAL_FRAMES = None  # например 600 для быстрой проверки; None = до конца
assert FRAME_STEP >= 1
cap.set(cv2.CAP_PROP_POS_FRAMES, bg_frames)
out_fps = fps / FRAME_STEP
writer = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (width,height))
if not writer.isOpened(): raise RuntimeError("Не удалось открыть output.mp4 для записи")
times_static = []; breakdown = []; detected_counts = []
index = bg_frames; processed = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok: break
        if (index - bg_frames) % FRAME_STEP == 0:
            t0 = time.perf_counter()
            r = detect_static(frame, background, THRESHOLD, KERNEL_SIZE, MIN_AREA)
            duration = time.perf_counter() - t0
            times_static.append(duration)
            breakdown.append(r["timings"])
            detected_counts.append(len(r["boxes"]))
            writer.write(r["annotated"])
            processed += 1
            if processed % 150 == 0: print(f"Обработано {processed} кадров")
            if MAX_EVAL_FRAMES is not None and processed >= MAX_EVAL_FRAMES: break
        index += 1
finally:
    writer.release()
if not times_static: raise RuntimeError("Нет кадров после 30-секундного фонового интервала")
print(f"output.mp4: {processed} кадров, FPS записи {out_fps:.2f}")
def describe_times(times, label):
    a = np.asarray(times) * 1000
    return dict(method=label, frames=len(a), mean_ms=a.mean(), min_ms=a.min(),
                max_ms=a.max(), algorithm_fps=1000/a.mean())
static_stats = describe_times(times_static, "Средний фон")
display(pd.DataFrame([static_stats]).round(3))
print("Сравнение с исходным видео:", "реальное время по вычислениям возможно" if static_stats["algorithm_fps"] >= fps else "скорости обработки недостаточно", f"({static_stats['algorithm_fps']:.1f} против {fps:.1f} FPS)")
stage_ms = pd.DataFrame(breakdown).mean().sort_values(ascending=False)*1000
display(stage_ms.rename("Среднее время, мс").to_frame())
stage_ms.plot.bar(figsize=(9,4), title="Затраты по этапам, CPU (среднее)")
plt.ylabel("мс / кадр"); plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

## 5. Сравнение с адаптивным MOG2
MOG2 строит адаптивную смесь гауссиан для каждого пикселя. Модель **предварительно прогревается теми же 30 секундами**, затем оценивается на последующих кадрах, чтобы условия сопоставления были одинаковыми. Для сравнения масок используем `detectShadows=False` и одинаковое морфологическое закрытие/фильтрацию площади. Времена MOG2 включают его `apply`, морфологию, компоненты и рисование, но исключают декодирование видео.

In [ ]:
def clean_and_boxes(frame, mask, kernel_size=KERNEL_SIZE, min_area=MIN_AREA):
    kernel = np.ones((kernel_size,kernel_size), np.uint8)
    closed = cv2.erode(cv2.dilate(mask,kernel),kernel)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    keep = [i for i in range(1,n) if stats[i,cv2.CC_STAT_AREA] >= min_area]
    clean = np.isin(labels,keep).astype(np.uint8)*255
    image = frame.copy()
    for i in keep:
        x,y,w,h,area = (int(v) for v in stats[i,:5])
        cv2.rectangle(image,(x,y),(x+w-1,y+h-1),(0,255,0),2)
    return clean, image, len(keep)

mog2 = cv2.createBackgroundSubtractorMOG2(history=bg_frames, varThreshold=16, detectShadows=False)
mog_cap = cv2.VideoCapture(VIDEO_PATH)
if not mog_cap.isOpened(): raise RuntimeError("Ошибка открытия видео для MOG2")
warmed = 0
while warmed < bg_frames:
    ok, frame = mog_cap.read()
    if not ok: break
    mog2.apply(frame, learningRate=-1)
    warmed += 1
if warmed != bg_frames: raise RuntimeError("Недостаточно кадров для MOG2")
times_mog2 = []; mog_examples = {}
index = bg_frames; processed = 0
try:
    while True:
        ok, frame = mog_cap.read()
        if not ok: break
        if (index-bg_frames) % FRAME_STEP == 0:
            t0 = time.perf_counter()
            mask = mog2.apply(frame, learningRate=-1)
            clean, annotated, count = clean_and_boxes(frame, mask)
            times_mog2.append(time.perf_counter()-t0)
            if index in sample_indices: mog_examples[index]=(clean.copy(), annotated.copy())
            processed += 1
            if MAX_EVAL_FRAMES is not None and processed >= MAX_EVAL_FRAMES: break
        index += 1
finally:
    mog_cap.release()
mog_stats = describe_times(times_mog2, "MOG2")
comparison = pd.DataFrame([static_stats,mog_stats])
display(comparison.round(3))
comparison.set_index("method")["algorithm_fps"].plot.bar(rot=0, ylabel="FPS (CPU)", title="Скорость обработки")
plt.axhline(fps,color="red",linestyle="--",label=f"Видео {fps:.1f} FPS")
plt.legend();plt.tight_layout();plt.show()
for i, frame, classic in zip(sample_indices,sample_frames,sample_results):
    if i not in mog_examples: continue
    m_mask, m_box = mog_examples[i]
    fig, axs=plt.subplots(2,2,figsize=(12,8))
    for ax, img, title in zip(axs.flat,
        [classic["mask"],m_mask,classic["annotated"],m_box],
        ["Усреднение: маска","MOG2: маска","Усреднение: рамки","MOG2: рамки"]):
        rgb_show(img,title,ax)
    fig.suptitle(f"Кадр {i}");plt.tight_layout();plt.show()

## Итоговый анализ и выводы
Автоматические метрики ниже рассчитываются **при запуске**. Без вручную размеченных истинных автомобилей нельзя честно вычислить precision/recall; визуальное сравнение позволяет оценить, например, ложные рамки от теней, пропуски и слипание машин.

- **Базовый метод** конкурентоспособен при неподвижной камере, постоянном освещении и редком движении; он прост, быстр и не обучается.
- **MOG2** обычно устойчивее при постепенной смене освещения и фоновых колебаниях, поскольку обновляет распределения значений пикселей. Но может терять долго стоящие автомобили, включив их в фон.
- У обоих методов рамка вокруг компоненты не является семантическим распознаванием автомобиля. Соседние автомобили могут сливаться.
- Среднее время этапов и фактическая пригодность к реальному времени зависят от видео, размера кадра и CPU.

При анализе укажите, какие ошибки видны на **ваших** кадрах. Значение FPS алгоритма не включает чтение/декодирование, сохранение и отображение видео.

In [ ]:
winner = "базовый метод" if static_stats["mean_ms"] < mog_stats["mean_ms"] else "MOG2"
print(f"Исходный поток: {fps:.2f} FPS")
print(f"Базовый метод: {static_stats['mean_ms']:.3f} мс/кадр, {static_stats['algorithm_fps']:.1f} FPS")
print(f"MOG2: {mog_stats['mean_ms']:.3f} мс/кадр, {mog_stats['algorithm_fps']:.1f} FPS")
print(f"Быстрее в этом эксперименте: {winner}.")
print(f"Самый дорогой этап базового метода: {stage_ms.index[0]} ({stage_ms.iloc[0]:.3f} мс).")
print("Вывод о реальном времени для базового метода:",
      "да (только вычисления)" if static_stats["algorithm_fps"] >= fps else "нет")
print("Вывод о реальном времени для MOG2:",
      "да (только вычисления)" if mog_stats["algorithm_fps"] >= fps else "нет")
comparison.to_csv("performance_comparison.csv", index=False)
print("Файлы: background.png, output.mp4, performance_comparison.csv")

In [ ]:
from google.colab import files
# Раскомментируйте желаемые строки после выполнения расчётов:
# files.download("background.png")
# files.download("output.mp4")
# files.download("performance_comparison.csv")